In [ ]:
import pandas as pd
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
df = pd.read_excel('/content/drive/MyDrive/Project_DA6401W/project_data_set.xlsx')
df.head()

,Unnamed: 0,match_id,date,innings,batting_team,bowling_team,over,ball,ball_no,batter,bowler,line,length,speed
0,218244,597999,2013-04-04,1,Royal Challengers Bangalore,Mumbai Indians,4,1,4.1,V Kohli,JJ Bumrah,1,2,1
1,218245,597999,2013-04-04,1,Royal Challengers Bangalore,Mumbai Indians,4,2,4.2,V Kohli,JJ Bumrah,2,3,3
2,218246,597999,2013-04-04,1,Royal Challengers Bangalore,Mumbai Indians,4,3,4.3,V Kohli,JJ Bumrah,2,3,3
3,218247,597999,2013-04-04,1,Royal Challengers Bangalore,Mumbai Indians,4,4,4.4,V Kohli,JJ Bumrah,2,2,1
4,218248,597999,2013-04-04,1,Royal Challengers Bangalore,Mumbai Indians,4,5,4.5,V Kohli,JJ Bumrah,1,3,3


#Sort data properly and Select required features

In [ ]:
df = df.sort_values(by=['match_id', 'over', 'ball']).reset_index(drop=True)
df = df[['match_id', 'over', 'ball', 'line', 'length', 'speed']]

#Normalize values

In [ ]:
scaler = MinMaxScaler()

df[['line', 'length', 'speed', 'over', 'ball']] = scaler.fit_transform(
    df[['line', 'length', 'speed', 'over', 'ball']]
)

#Create sequences

In [ ]:
sequence_length = 6

X = []
y = []

features = df[['line', 'length', 'speed', 'over', 'ball']].values

for i in range(len(features) - sequence_length):
    X.append(features[i:i+sequence_length])
    y.append(features[i+sequence_length][:3])  # predict line, length, speed

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3589, 6, 5)
y shape: (3589, 3)


#Train-test split

In [ ]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

#Build RNN model

In [ ]:
rnn_model = Sequential([
    SimpleRNN(64, input_shape=(X.shape[1], X.shape[2])),
    Dense(32, activation='relu'),
    Dense(3)
])

rnn_model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


#Build LSTM model


In [ ]:
lstm_model = Sequential([
    LSTM(64, input_shape=(X.shape[1], X.shape[2])),
    Dense(32, activation='relu'),
    Dense(3)
])

lstm_model.compile(optimizer='adam', loss='mse')

#Build GRU model

In [ ]:
gru_model = Sequential([
    GRU(64, input_shape=(X.shape[1], X.shape[2])),
    Dense(32, activation='relu'),
    Dense(3)
])

gru_model.compile(optimizer='adam', loss='mse')

#Train models

In [ ]:
rnn_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
lstm_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))
gru_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.1930 - val_loss: 0.1743
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1695 - val_loss: 0.1709
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1670 - val_loss: 0.1708
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1651 - val_loss: 0.1714
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1656 - val_loss: 0.1758
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1654 - val_loss: 0.1740
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1652 - val_loss: 0.1695
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1647 - val_loss: 0.1714
Epoch 9/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1627 - val_loss: 0.1700
Epoch 10/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1626 - val_loss: 0.1702
Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.1884 - val_loss: 0.1670
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1676 - val_los

#Evaluation function

In [ ]:
def evaluate_model(model, name):
    pred = model.predict(X_test)

    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    print(f"{name} Results")
    print("MSE:", mse)
    print("R2:", r2)
    print("-"*30)

#Compare models

In [ ]:
evaluate_model(rnn_model, "RNN")
evaluate_model(lstm_model, "LSTM")
evaluate_model(gru_model, "GRU")

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
RNN Results
MSE: 0.17020942766521927
R2: -0.03948221775482982
------------------------------
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
LSTM Results
MSE: 0.16496483315598917
R2: -0.007471159042574597
------------------------------
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
GRU Results
MSE: 0.16615749018237802
R2: -0.014754796910272594
------------------------------


#MSE (Mean Squared Error)

Lower = better

Model Values:

RNN → worst

GRU → slightly better

LSTM → best

#R² score meaning:
1.0 → perfect prediction
0.0 → model is as bad as guessing average
< 0 → model is worse than guessing average

#R² is negative for ALL models

LSTM is slightly better than RNN and GRU

But overall model has no strong predictive power yet

Data/features need improvement

#Although LSTM slightly outperformed RNN and GRU, all models show negative R² indicating that raw ball-level features alone are insufficient to fully capture bowling decision complexity.

#Rolling prediction function

In [ ]:
def rollout(model, sequence, steps=18):
    """
    sequence = first 6 balls (shape: 6 x features)
    steps = how many balls to predict
    """

    seq = sequence.copy()
    preds = []

    for _ in range(steps):

        inp = seq.reshape(1, seq.shape[0], seq.shape[1])

        pred = model.predict(inp, verbose=0)[0]
        preds.append(pred)

        # create next input (we use prediction here)
        new_row = np.zeros(seq.shape[1])
        new_row[:3] = pred

        seq = np.vstack([seq[1:], new_row])

    return np.array(preds)

# Pick one match (24 balls)

In [ ]:
match_sample = df[df['match_id'] == df['match_id'].iloc[0]]

match_data = match_sample[['line','length','speed','over','ball']].values

initial_sequence = match_data[:6]   # first 6 balls
actual_sequence = match_data[6:24]  # next 18 balls

# Get predictions from all models

In [ ]:
rnn_pred = rollout(rnn_model, initial_sequence)
lstm_pred = rollout(lstm_model, initial_sequence)
gru_pred = rollout(gru_model, initial_sequence)

# Create comparison table

In [ ]:

comparison = pd.DataFrame({
    "Actual_line": actual_sequence[:,0],
    "RNN_line": rnn_pred[:,0],
    "LSTM_line": lstm_pred[:,0],
    "GRU_line": gru_pred[:,0],

    "Actual_length": actual_sequence[:,1],
    "RNN_length": rnn_pred[:,1],
    "LSTM_length": lstm_pred[:,1],
    "GRU_length": gru_pred[:,1],

    "Actual_speed": actual_sequence[:,2],
    "RNN_speed": rnn_pred[:,2],
    "LSTM_speed": lstm_pred[:,2],
    "GRU_speed": gru_pred[:,2],
})

comparison.head()

,Actual_line,RNN_line,LSTM_line,GRU_line,Actual_length,RNN_length,LSTM_length,GRU_length,Actual_speed,RNN_speed,LSTM_speed,GRU_speed
0,1.0,0.532913,0.513245,0.513938,0.0,0.482170,0.524147,0.508755,0.0,0.520510,0.493835,0.450413
1,0.0,0.436264,0.502881,0.501852,1.0,0.455244,0.512695,0.496385,0.0,0.495300,0.500683,0.452437
2,1.0,0.596074,0.489219,0.509195,1.0,0.563767,0.491954,0.474712,1.0,0.344824,0.493534,0.463778
3,1.0,0.607050,0.469509,0.502677,0.5,0.499856,0.466713,0.447402,0.5,0.563486,0.477659,0.465229
4,0.5,0.551515,0.453954,0.486106,0.0,0.521608,0.451617,0.422495,0.0,0.379015,0.460808,0.453974


# Create conversion function

In [ ]:
def to_category(x):
    if x < 0.25:
        return 0
    elif x < 0.50:
        return 0.5
    elif x < 0.75:
        return 1
    else:
        return 2

# Apply conversion to predictions

In [ ]:
# convert RNN
rnn_line_cat = np.array([to_category(x) for x in rnn_pred[:,0]])
rnn_length_cat = np.array([to_category(x) for x in rnn_pred[:,1]])
rnn_speed_cat = np.array([to_category(x) for x in rnn_pred[:,2]])

# convert LSTM
lstm_line_cat = np.array([to_category(x) for x in lstm_pred[:,0]])
lstm_length_cat = np.array([to_category(x) for x in lstm_pred[:,1]])
lstm_speed_cat = np.array([to_category(x) for x in lstm_pred[:,2]])

# convert GRU
gru_line_cat = np.array([to_category(x) for x in gru_pred[:,0]])
gru_length_cat = np.array([to_category(x) for x in gru_pred[:,1]])
gru_speed_cat = np.array([to_category(x) for x in gru_pred[:,2]])

# Build corrected comparison table

In [ ]:
def highlight_correct(df):
    def style_row(row):
        styles = []

        for col in df.columns:
            if col.startswith("RNN") or col.startswith("LSTM") or col.startswith("GRU"):
                # extract feature type (line/length/speed)
                feature = col.split("_")[1]
                actual_col = "Actual_" + feature

                if row[col] == row[actual_col]:
                    styles.append("background-color: lightgreen")
                else:
                    styles.append("background-color: lightcoral")
            else:
                styles.append("")  # no color for actual columns

        return styles

    return df.style.apply(style_row, axis=1)

In [ ]:
comparison = pd.DataFrame({
    "Actual_line": actual_sequence[:,0],
    "RNN_line": rnn_line_cat,
    "LSTM_line": lstm_line_cat,
    "GRU_line": gru_line_cat,

    "Actual_length": actual_sequence[:,1],
    "RNN_length": rnn_length_cat,
    "LSTM_length": lstm_length_cat,
    "GRU_length": gru_length_cat,

    "Actual_speed": actual_sequence[:,2],
    "RNN_speed": rnn_speed_cat,
    "LSTM_speed": lstm_speed_cat,
    "GRU_speed": gru_speed_cat,
})

highlight_correct(comparison.head(20))

,Actual_line,RNN_line,LSTM_line,GRU_line,Actual_length,RNN_length,LSTM_length,GRU_length,Actual_speed,RNN_speed,LSTM_speed,GRU_speed
0,1.000000,1.000000,1.000000,1.000000,0.000000,0.500000,1.000000,1.000000,0.000000,1.000000,0.500000,0.500000
1,0.000000,0.500000,1.000000,1.000000,1.000000,0.500000,1.000000,0.500000,0.000000,0.500000,1.000000,0.500000
2,1.000000,1.000000,0.500000,1.000000,1.000000,1.000000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000
3,1.000000,1.000000,0.500000,1.000000,0.500000,0.500000,0.500000,0.500000,0.500000,1.000000,0.500000,0.500000
4,0.500000,1.000000,0.500000,0.500000,0.000000,1.000000,0.500000,0.500000,0.000000,0.500000,0.500000,0.500000
5,0.000000,1.000000,0.500000,0.500000,0.500000,1.000000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000
6,0.000000,1.000000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000
7,0.500000,1.000000,0.500000,0.500000,0.000000,0.500000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000
8,0.000000,0.500000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
9,0.000000,1.000000,0.500000,0.500000,0.000000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000


#Live data

In [ ]:
import pandas as pd

df = pd.DataFrame([
    [1, 1, "Abhishek Sharma", "Jasprit Bumrah", 2, 2, 2],
    [1, 2, "Abhishek Sharma", "Jasprit Bumrah", 1, 3, 2],
    [1, 3, "Travis Head", "Jasprit Bumrah", 0, 2, 2],
    [1, 4, "Abhishek Sharma", "Jasprit Bumrah", 2, 1, 1],
    [1, 4, "Abhishek Sharma", "Jasprit Bumrah", 0, 3, 3],
    [1, 4, "Abhishek Sharma", "Jasprit Bumrah", 2, 1, 3],
    [1, 5, "Abhishek Sharma", "Jasprit Bumrah", 1, 1, 2],
    [1, 6, "Abhishek Sharma", "Jasprit Bumrah", 2, 1, 1],
], columns=[
    "over","ball","batter","bowler","line","length","speed"
])

In [ ]:
features = df[['line','length','speed','over','ball']].values

In [ ]:
seq_len = 5  # must match training

X_test = []
y_test = []

for i in range(len(features) - seq_len):
    X_test.append(features[i:i+seq_len])
    y_test.append(features[i+seq_len][:3])  # predict only line/length/speed

X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:
rnn_pred = rnn_model.predict(X_test)
lstm_pred = lstm_model.predict(X_test)
gru_pred = gru_model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step


In [ ]:
def to_class(x):
    if x < 0.33:
        return 0
    elif x < 0.66:
        return 1
    else:
        return 2

In [ ]:
comparison = pd.DataFrame({
    "Actual_line": y_test[:,0],
    "RNN_line": [to_class(x) for x in rnn_pred[:,0]],
    "LSTM_line": [to_class(x) for x in lstm_pred[:,0]],
    "GRU_line": [to_class(x) for x in gru_pred[:,0]],

    "Actual_length": y_test[:,1],
    "RNN_length": [to_class(x) for x in rnn_pred[:,1]],
    "LSTM_length": [to_class(x) for x in lstm_pred[:,1]],
    "GRU_length": [to_class(x) for x in gru_pred[:,1]],

    "Actual_speed": y_test[:,2],
    "RNN_speed": [to_class(x) for x in rnn_pred[:,2]],
    "LSTM_speed": [to_class(x) for x in lstm_pred[:,2]],
    "GRU_speed": [to_class(x) for x in gru_pred[:,2]],
})

comparison

,Actual_line,RNN_line,LSTM_line,GRU_line,Actual_length,RNN_length,LSTM_length,GRU_length,Actual_speed,RNN_speed,LSTM_speed,GRU_speed
0,2,1,1,2,1,1,2,2,3,1,1,1
1,1,2,2,2,1,1,2,2,2,1,1,1
2,2,2,2,2,1,0,2,2,1,1,2,2
